# 01 — Data-quality audit of the NEPSE floorsheet corpus (2021–2024)

**EXPERIMENT EXP-0002f** (register: `docs/experiment_register.md`) · Phase 2.
Corpus: **57,927,926 raw rows → 57,892,630 after determinstic dedupe**
(`floorsheet_dedup`), 938 of 940 calendar sessions (99.79%; the 2 gap
sessions have no known source — documented, non-silent).

Every number in this notebook is **[REAL]** — computed live from
`data/duckdb/nepse.duckdb` or the exported audit tables in
`outputs/tables/`. Audit derives from EXP-0002b (ingestion),
EXP-0002c (audit), EXP-0002d/e (OHLCV + reconciliation).

In [1]:
import sys
from pathlib import Path
import pandas as pd

_root = Path.cwd()
while not (_root / "configs").exists():
    if _root.parent == _root:
        raise RuntimeError("configs/ not found above cwd")
    _root = _root.parent
ROOT = _root
sys.path.insert(0, str(ROOT / "src"))

dup    = pd.read_csv(ROOT / "outputs/tables/corpus_duplicate_keys.csv")
nobrk  = pd.read_csv(ROOT / "outputs/tables/corpus_missing_broker_ids.csv")
lowact = pd.read_csv(ROOT / "outputs/tables/corpus_low_activity_days.csv")
daily  = pd.read_csv(ROOT / "outputs/tables/corpus_daily_summary.csv")
syms   = pd.read_csv(ROOT / "outputs/tables/corpus_symbol_coverage.csv")
recon  = pd.read_csv(ROOT / "outputs/tables/volume_reconciliation.csv")

print("audit tables loaded:",
      {n: len(d) for n, d in [("duplicate_keys", dup), ("missing_broker", nobrk),
                              ("low_activity", lowact), ("daily_summary", daily),
                              ("symbol_coverage", syms), ("volume_recon", recon)]})

audit tables loaded: {'duplicate_keys': 752, 'missing_broker': 19, 'low_activity': 114167, 'daily_summary': 938, 'symbol_coverage': 530, 'volume_recon': 11283}


## Dedupe classes (EXP-0002c A1/A2)
Dedupe key = (date, symbol, contract_no); page-split duplicates collapse only when the FULL row signature matches. Signature-conflicting keys are **documented, never silently merged** (deterministic order-by pick).

In [2]:
n_exact = int((dup["n_signatures"] == 1).sum())
n_conf  = int((dup["n_signatures"] > 1).sum())
print(f"duplicate (date,symbol,contract_no) keys: {len(dup):,} "
      f"| exact page-dups: {n_exact:,} | signature-conflicts (NOT merged): {n_conf:,} [REAL]")
dup["n_signatures"].value_counts().rename("n_keys").head()

duplicate (date,symbol,contract_no) keys: 752 | exact page-dups: 506 | signature-conflicts (NOT merged): 246 [REAL]


n_signatures
1     506
6       5
54      4
2       4
3       4
Name: n_keys, dtype: int64

## Missing broker ids & low-activity symbol-days (A3/A5)
Missing-broker pairs are **junk scrape lines** in the upstream source (symbol present, buyer/seller/qty/rate/amount all empty) — they stay in `floorsheet_raw` with NULL volume so no aggregation touches them. Low-activity (<100 trades) pairs are suspensions/illiquid scrips: flagged, governed at feature time by the DC2 coverage rule, not deleted.

In [3]:
print(f"missing-broker (symbol,date) pairs: {len(nobrk):,} [REAL]  (junk scrape rows)")
print(f"(symbol,date) pairs with <100 trades: {len(lowact):,} "
      f"(min {lowact['n_trades'].min()}) [REAL]")
nobrk.head()

missing-broker (symbol,date) pairs: 19 [REAL]  (junk scrape rows)
(symbol,date) pairs with <100 trades: 114,167 (min 1) [REAL]


,date,symbol,n_nobuyer,n_noseller
0,2021-08-19,BOKL,1,1
1,2021-08-08,GBIME,1,1
2,2021-08-02,NaN,1,1
3,2021-08-12,ICFC,1,1
4,2021-07-26,DHPL,1,1


## Source spans (A8) — live DuckDB aggregate over the raw table

In [4]:
import duckdb
from nepsemsc.data.duckdb_loader import get_conn

con = get_conn()
src = con.execute("""
    SELECT source, COUNT(*) AS rows, COUNT(DISTINCT date) AS days,
           MIN(date) AS first_day, MAX(date) AS last_day
    FROM floorsheet_raw GROUP BY source ORDER BY source
""").df()
print('total raw rows:', f"{int(src['rows'].sum()):,}", '[REAL]')
src

total raw rows: 57,927,926 [REAL]


,source,rows,days,first_day,last_day
0,kaggle_viking303,36706934,706,2021-01-03,2023-12-28
1,socrateai,21220992,232,2024-01-01,2024-12-31


## Calendar coverage & the 2 documented gaps
938 of 940 DC3 calendar sessions ingested (99.79%). Gap sessions **2021-09-01** and **2023-11-27** exist in the OHLCV calendar but no floorsheet day file exists in any verified source — logged in `outputs/tables/ingestion_session_gaps.csv`, never imputed.

In [5]:
gaps = con.execute("""
    WITH cal AS (SELECT CAST(session_date AS DATE) AS date
                 FROM read_parquet('{}')),
         ing AS (SELECT DISTINCT date FROM floorsheet_raw)
    SELECT c.date FROM cal c LEFT JOIN ing i USING (date)
    WHERE i.date IS NULL ORDER BY 1
""".format(ROOT / "data/processed/nepse_calendar.parquet")).df()
by_year = con.execute("""
    SELECT year(date) AS yr, COUNT(DISTINCT date) AS sessions, COUNT(*) AS rows
    FROM floorsheet_raw GROUP BY 1 ORDER BY 1
""").df()
print("gap sessions:", ", ".join(gaps["date"].astype(str)), "[REAL]")
by_year

gap sessions: 2021-09-01, 2023-11-27 [REAL]


,yr,sessions,rows
0,2021,238,17769216
1,2022,242,8686330
2,2023,226,10251388
3,2024,232,21220992


## The one relabel (EXP-0002b) — live re-check
Kaggle file `2021_03_07_floorsheet.csv` is ingested **as 2021-03-08**. Evidence triple: no 2021-03-07 session in either OHLCV source; CGH rate range [635, 665] == tier1 2021-03-08 OHLC band; CGH quantity sum 910 == tier1 volume 910. Re-check the quantity identity live:

In [6]:
cgh = con.execute("""
    SELECT date, SUM(volume) AS cgh_qty, COUNT(DISTINCT contract_no) AS ids
    FROM floorsheet_raw WHERE symbol = 'CGH' AND date BETWEEN DATE '2021-03-06' AND DATE '2021-03-09'
    GROUP BY date ORDER BY date
""").df()
tier1_vol_0308 = 910  # tier1_kaggle CGH.csv 2021-03-08 Volume (EXP-0002b evidence)
ok = int(cgh.loc[cgh["date"].astype(str) == "2021-03-08", "cgh_qty"].iloc[0]) == tier1_vol_0308
print(f"CGH 2021-03-08 corpus qty sum == tier1 volume 910: {ok} [REAL]")
cgh

CGH 2021-03-08 corpus qty sum == tier1 volume 910: True [REAL]


,date,cgh_qty,ids
0,2021-03-08,910.0,94
1,2021-03-09,1629.0,164


## Source-side defect: 2023-02-16 (declared BEFORE Phase 3)
All 35,043 rows of that day share one transaction id — the upstream file left Transact.-No. unpopulated. Level aggregates keep the day; any **ordering-dependent** feature MUST exclude it (`contract_no_defect` flag).

In [7]:
degen = con.execute("""
    WITH per_day AS (
      SELECT date, COUNT(DISTINCT contract_no)::DOUBLE / COUNT(*) AS id_share,
             COUNT(*) AS n_rows, COUNT(DISTINCT contract_no) AS n_ids
      FROM floorsheet_raw GROUP BY date)
    SELECT * FROM per_day WHERE id_share < 0.5 ORDER BY id_share
""").df()
print("days with id_share < 0.5:", len(degen), "[REAL]")
degen

days with id_share < 0.5: 1 [REAL]


,date,id_share,n_rows,n_ids
0,2023-02-16,0.000029,35043,1


## Volume reconciliation vs OHLCV (EXP-0002e)
Top-12 universe, windows 2021-2024. Median `volume_ratio` = 1.0000 (buyer-side counting; CGH 910 = 910 confirms). The full-market ~2x spot ratio is a **scope artifact**: tier1 OHLCV covers 100 of 530 corpus scrips — a limitation, not a finding.

In [8]:
con.close()
both = recon.dropna(subset=["volume_ratio"])
med = both["volume_ratio"].median()
b1  = both["volume_ratio"].between(0.95, 1.25).mean()
b2  = both["volume_ratio"].between(0.98, 1.02).mean()
print(f"pairs both-sides: {len(both):,} | median volume_ratio {med:.4f} | "
      f"in [0.95,1.25]: {b1:.1%} | in [0.98,1.02]: {b2:.1%} [REAL]")
syms.head(12)[["symbol", "n_dates", "first_date", "last_date"]]

pairs both-sides: 10,909 | median volume_ratio 1.0000 | in [0.95,1.25]: 96.6% | in [0.98,1.02]: 94.7% [REAL]


,symbol,n_dates,first_date,last_date
0,TRH,934,2021-01-03,2024-12-31
1,STC,934,2021-01-03,2024-12-31
2,SHL,933,2021-01-03,2024-12-31
3,OHL,932,2021-01-03,2024-12-31
4,UMRH,930,2021-01-03,2024-12-31
5,AKPL,930,2021-01-03,2024-12-31
6,SJCL,930,2021-01-03,2024-12-31
7,HPPL,930,2021-01-03,2024-12-31
8,AHPC,930,2021-01-03,2024-12-31
9,CHCL,930,2021-01-03,2024-12-31


## Take-aways (Phase 2 → Phase 3 contract)

1. Corpus **[REAL]**: 57,927,926 raw rows / 57,892,630 dedup / 938 sessions / 530 symbols.
2. Two sessions have **no source** (2021-09-01, 2023-11-27) — non-imputable, documented.
3. One deliberate **relabel** with a 3-leg evidence chain (2021-03-07→03-08).
4. **2023-02-16** = `contract_no_defect`: excludes only ordering-dependent statistics.
5. Junk scrape rows carry NULL volume — excluded by construction from aggregates.
6. Numeric sanity: negative volume 0, negative amount 0; non-positive price 8 and
   NULL contract_no 2 (all inside defective/junk classes inspected in EXP-0002c).
7. Phase 3 feature gates honoured: junk/NULL filtering, source propagation,
   per-fold universe re-selection (DC1), 2023-02-16 ordering exclusion.